# EmpowerLens — D15 truncation test (Kaggle GPU)

Trains MentalRoBERTa (D12 winner) on **multiclass** at **max length 256** with two truncation options —
**head** (keep the first tokens) and **head_tail** (keep the first 128 + the end) — for seeds 42, 1337, 2024
(6 runs, D09/T14 settings: lr 3e-5, 5 epochs, weight decay 0.1, batch 16, `--deterministic`).
Val only: `src/evaluate.py` is not run and test is never read. Also measures the share of texts cut at 256 and 512.
Output: `results_truncation/` (meta.json + epoch_history.csv per run, token_lengths.json), ~30–60 min on a T4/P100.

**Before running**
1. Settings → **Accelerator: GPU (T4 or P100)**, **Internet: On**.
2. `mental/mental-roberta-base` is gated: accept its terms on huggingface.co and add a Kaggle secret
   **`HF_TOKEN`** (Add-ons → Secrets) with a read token.

**Resumable.** Re-running the loop cell skips every (option, seed) whose meta.json is already in
`/kaggle/working/results_truncation/`. Checkpoints live in `/kaggle/temp` and are deleted after each run.
When it finishes, download `results_truncation.zip`, unzip it into `results/results_truncation/` in the repo,
and run `python -m src.truncation_report --results results/results_truncation`.

In [ ]:
# 1. HF token, single GPU, clone the repo into /kaggle/temp (not saved with the output), install.
import os
from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded.")
except Exception as e:
    print(f"[warn] no HF_TOKEN secret ({e}) - mental-roberta-base will fail to download.")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # avoid multi-GPU load hangs on T4 x2

REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "lumia-space"
!rm -rf /kaggle/temp/empowerlens && mkdir -p /kaggle/temp && git clone --depth 1 --branch $BRANCH $REPO_URL /kaggle/temp/empowerlens
%cd /kaggle/temp/empowerlens
!pip install -q -r requirements-transformer.txt
!ls src/truncation_report.py data/splits && python -c "import torch; print('cuda:', torch.cuda.is_available())"

In [ ]:
# 2. Shared config + helpers (re-run this cell first after any kernel restart).
import os, json, shutil, subprocess, time
from pathlib import Path
os.chdir("/kaggle/temp/empowerlens")

MODEL  = "mental/mental-roberta-base"
TAG    = MODEL.split("/")[-1]
SEEDS  = (42, 1337, 2024)
ARMS   = ("head", "head_tail")
RES    = Path("/kaggle/working/results_truncation")
CKPT   = Path("/kaggle/temp/ckpt_d15")
RECIPE = ("--task multiclass --max-length 256 --lr 3e-5 --epochs 5 --weight-decay 0.1 "
          "--batch-size 16 --deterministic --splits data/splits")

def sh(cmd):
    print(f"
$ {cmd}", flush=True)
    t = time.time()
    subprocess.run(cmd, shell=True, check=True)
    print(f"[done in {(time.time() - t) / 60:.1f} min]", flush=True)

def meta_file(arm, seed):
    return RES / arm / f"seed_{seed}" / "meta.json"

def train(arm, seed):
    # Separate --out per arm: run_name has no truncation in it, so the arms would overwrite each other.
    sh(f"python -m src.train_transformer --model {MODEL} --seed {seed} --truncation {arm} "
       f"--out {CKPT}/{arm} {RECIPE}")
    return CKPT / arm / f"multiclass_{TAG}_{seed}"

def keep_meta(ckpt, arm, seed):
    dst = RES / arm / f"seed_{seed}"
    dst.mkdir(parents=True, exist_ok=True)
    for f in ("meta.json", "epoch_history.csv"):
        if (ckpt / f).exists():
            shutil.copy(ckpt / f, dst / f)
print("config ok:", MODEL, SEEDS, ARMS)

In [ ]:
# 3. Smoke test (~2 min): tiny head_tail run at 256. Writes nothing to RES.
sh(f"python -m src.train_transformer --task multiclass --model {MODEL} --seed 42 --smoke "
   f"--max-length 256 --truncation head_tail --out /kaggle/temp/smoke")
m = json.loads(Path(f"/kaggle/temp/smoke/multiclass_{TAG}_42_smoke/meta.json").read_text())
assert m["truncation"] == "head_tail" and m["max_length"] == 256
print("SMOKE OK - val macro_f1_10", m["val_metrics"]["eval_macro_f1_10"])
shutil.rmtree("/kaggle/temp/smoke", ignore_errors=True)

In [ ]:
# 4. Real runs: 2 options x 3 seeds = 6 trainings. Resumable.
for seed in SEEDS:
    for arm in ARMS:
        if meta_file(arm, seed).exists():
            print(f"skip {arm} {seed} (done)"); continue
        ck = train(arm, seed)
        keep_meta(ck, arm, seed)
        shutil.rmtree(ck, ignore_errors=True)
    print(f"===== seed {seed} done =====", flush=True)

missing = [f"{a}/{s}" for a in ARMS for s in SEEDS if not meta_file(a, s).exists()]
print("MISSING:", missing if missing else "none - all 6 runs present")

In [ ]:
# 5. Share of texts cut at 256 and 512, with the real MentalRoBERTa tokenizer.
#    Text lengths only (no labels). 'corpus' = all 2,530 rows, the basis of the earlier 3.2% figure.
import statistics
from transformers import AutoTokenizer
from src.data import load_raw
from src.train_transformer import load_split
from src.truncation_report import token_lengths, cut_share

tok = AutoTokenizer.from_pretrained(MODEL)
parts = {"corpus": load_raw()["Patient Question"],
         "train": load_split("data/splits", "train")["Patient Question"],
         "val":   load_split("data/splits", "val")["Patient Question"]}
out = {}
for name, texts in parts.items():
    L = token_lengths(texts.astype(str), tok)
    out[name] = {"n": len(L), "median": statistics.median(L), "max": max(L),
                 "cut_256": cut_share(L, 256), "cut_512": cut_share(L, 512)}
    print(name, out[name])
RES.mkdir(parents=True, exist_ok=True)
(RES / "token_lengths.json").write_text(json.dumps(out, indent=2))

In [ ]:
# 6. Quick look + zip for download.
!python -m src.truncation_report --results {RES}
!cd /kaggle/working && rm -f results_truncation.zip && zip -qr results_truncation.zip results_truncation
!du -sh /kaggle/working/results_truncation && ls /kaggle/working